# Module 29 — Two-Tower Neural Network for Recommendation Retrieval

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: PyTorch, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 28 (Siamese BERT), Module 5 (Custom DataLoader)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic User-Item Interactions](#3-synthetic-user-item-interactions)
4. [Two-Tower Architecture](#4-two-tower-architecture)
5. [Training with In-Batch Negatives](#5-training-with-in-batch-negatives)
6. [Retrieval Evaluation](#6-retrieval-evaluation)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why two-tower networks for recommendations?

Two-tower architectures enable **efficient retrieval** at scale:
- **Separate encoders**: User and item towers encode independently.
- **Fast retrieval**: Pre-compute item embeddings, query with user embedding.
- **Scalable**: Handle millions of items with approximate nearest neighbors.

**Applications at Yandex**:
1. **Content recommendation**: Recommend articles, videos to users.
2. **Ad targeting**: Match ads to user interests.
3. **Search ranking**: Retrieve relevant documents efficiently.

**Key advantages**:
- **Efficiency**: O(1) retrieval with pre-computed embeddings.
- **Flexibility**: Can incorporate various user/item features.
- **Scalability**: Works with millions of items.

In this notebook we will:
1. Generate synthetic user-item interaction logs.
2. Build a two-tower neural network.
3. Train with in-batch negatives.
4. Evaluate with recall@K.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Deep learning ────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ All imports loaded")
print(f"  Device: {device}")

---
## §3 · Synthetic User-Item Interactions

In [ ]:
# Generate synthetic user-item interactions
N_USERS = 10000
N_ITEMS = 5000
N_INTERACTIONS = 100000

# Generate interactions
interactions = pd.DataFrame({
    'user_id': np.random.randint(0, N_USERS, N_INTERACTIONS),
    'item_id': np.random.randint(0, N_ITEMS, N_INTERACTIONS),
    'rating': np.random.choice([0, 1], N_INTERACTIONS, p=[0.7, 0.3]),
    'timestamp': pd.date_range('2024-01-01', periods=N_INTERACTIONS, freq='1min')
})

# Remove duplicates
interactions = interactions.drop_duplicates(subset=['user_id', 'item_id'])

print(f"✓ Generated {len(interactions)} user-item interactions")
print(f"  Users: {N_USERS}")
print(f"  Items: {N_ITEMS}")
print(f"  Positive interactions: {interactions['rating'].sum()}")
print(f"  Interaction rate: {interactions['rating'].mean()*100:.1f}%")

---
## §4 · Two-Tower Architecture

In [ ]:
# Define two-tower model
class UserTower(nn.Module):
    def __init__(self, n_users, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(n_users, embedding_dim)
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
    
    def forward(self, user_ids):
        x = self.embedding(user_ids)
        return self.fc(x)

class ItemTower(nn.Module):
    def __init__(self, n_items, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(n_items, embedding_dim)
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
    
    def forward(self, item_ids):
        x = self.embedding(item_ids)
        return self.fc(x)

class TwoTowerModel(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64):
        super().__init__()
        self.user_tower = UserTower(n_users, embedding_dim)
        self.item_tower = ItemTower(n_items, embedding_dim)
    
    def forward(self, user_ids, item_ids):
        user_emb = self.user_tower(user_ids)
        item_emb = self.item_tower(item_ids)
        return user_emb, item_emb

# Create model
model = TwoTowerModel(N_USERS, N_ITEMS).to(device)

print(f"✓ Two-tower model created")
print(f"  User tower parameters: {sum(p.numel() for p in model.user_tower.parameters()):,}")
print(f"  Item tower parameters: {sum(p.numel() for p in model.item_tower.parameters()):,}")

---
## §5 · Training with In-Batch Negatives

In [ ]:
# Define loss function with in-batch negatives
class InBatchNegativesLoss(nn.Module):
    def __init__(self, temperature=0.1):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, user_emb, item_emb):
        # Normalize embeddings
        user_emb = nn.functional.normalize(user_emb, dim=1)
        item_emb = nn.functional.normalize(item_emb, dim=1)
        
        # Compute similarity matrix
        logits = torch.matmul(user_emb, item_emb.T) / self.temperature
        
        # Labels: diagonal is positive
        labels = torch.arange(len(logits), device=logits.device)
        
        # Cross-entropy loss
        loss = nn.functional.cross_entropy(logits, labels)
        return loss

# Training setup
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = InBatchNegativesLoss(temperature=0.1)

N_EPOCHS = 5
BATCH_SIZE = 256

print(f"Training setup:")
print(f"  Epochs: {N_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Loss: In-batch negatives contrastive loss")
print(f"  Temperature: 0.1")

---
## §6 · Retrieval Evaluation

In [ ]:
# Evaluate retrieval performance
def compute_recall_at_k(model, test_data, k=10):
    """Compute recall@K for the model."""
    model.eval()
    
    # Get all item embeddings
    all_items = torch.arange(N_ITEMS, device=device)
    with torch.no_grad():
        _, item_embeddings = model(all_items, all_items)
    
    recalls = []
    for user_id in test_data['user_id'].unique()[:100]:  # Sample users
        # Get user embedding
        user_tensor = torch.tensor([user_id], device=device)
        with torch.no_grad():
            user_embedding, _ = model(user_tensor, user_tensor)
        
        # Get relevant items
        relevant_items = set(
            test_data[(test_data['user_id'] == user_id) & (test_data['rating'] == 1)]['item_id']
        )
        
        if len(relevant_items) == 0:
            continue
        
        # Compute similarities
        similarities = torch.matmul(user_embedding, item_embeddings.T).squeeze()
        top_k_items = similarities.argsort(descending=True)[:k].cpu().numpy()
        
        # Compute recall
        hits = len(set(top_k_items) & relevant_items)
        recall = hits / min(len(relevant_items), k)
        recalls.append(recall)
    
    return np.mean(recalls)

# Compute recall (using random model for demo)
recall_10 = compute_recall_at_k(model, interactions, k=10)

print("=" * 70)
print("RETRIEVAL EVALUATION")
print("=" * 70)
print(f"\nRecall@10: {recall_10:.4f}")
print(f"\nNote: This is using a randomly initialized model.")
print(f"After training, recall@10 would typically be 0.3-0.5.")

---
## §7 · Visualization

In [ ]:
# Visualize embedding space
# Get sample embeddings
sample_users = torch.arange(100, device=device)
sample_items = torch.arange(100, device=device)

with torch.no_grad():
    user_embs, item_embs = model(sample_users, sample_items)

# Convert to numpy
user_embs_np = user_embs.cpu().numpy()
item_embs_np = item_embs.cpu().numpy()

# Create visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=user_embs_np[:, 0],
    y=user_embs_np[:, 1],
    mode='markers',
    name='Users',
    marker=dict(color='#636EFA', size=8)
))

fig.add_trace(go.Scatter(
    x=item_embs_np[:, 0],
    y=item_embs_np[:, 1],
    mode='markers',
    name='Items',
    marker=dict(color='#EF553B', size=8)
))

fig.update_layout(
    title='Two-Tower Embedding Space',
    xaxis_title='Dimension 1',
    yaxis_title='Dimension 2',
    height=500
)
fig.show()

print("💡 Key insights:")
print("   - Users and items exist in same embedding space")
print("   - Similar users/items should cluster together")
print("   - Retrieval = find nearest items to user embedding")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Recommendation Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Large-scale recommendation, ad targeting, content matching |
> | **Architecture** | Separate user/item towers with shared embedding space |
> | **Training** | In-batch negatives for efficient contrastive learning |
> | **Evaluation** | Recall@K, NDCG@K, MRR |
> | **Deployment** | Pre-compute item embeddings, use FAISS for retrieval |
>
> ### 💡 Yandex-Specific Guidelines
>
> 1. **Production pipeline**:
>    ```
>    User request → Encode user → FAISS retrieval → Re-rank → Display
>    ```
>
> 2. **Scalability considerations**:
>    | Scale | Items | Latency | Method |
>    |-------|-------|---------|--------|
>    | Small | < 100K | < 10ms | Exact search |
>    | Medium | 100K-10M | < 50ms | FAISS IVF |
>    | Large | > 10M | < 100ms | FAISS HNSW |
>
> 3. **Training tips**:
>    - Use hard negatives for better discrimination
>    - Apply temperature tuning for contrastive loss
>    - Monitor embedding norms and collapse
>
> ### 🔑 Key Takeaways
>
> 1. **Two-tower architecture** enables efficient retrieval at scale.
> 2. **In-batch negatives** provide efficient contrastive training.
> 3. **Pre-compute item embeddings** for fast real-time retrieval.
> 4. **FAISS** is essential for approximate nearest neighbor search.
> 5. **Recall@K** is the primary metric for retrieval quality.